# ISEA Session 5 — Digital Forensics in Google Colab
## From First Cell to Explainable Verdict

**Series:** Intelligent Digital Forensics: LLMs, NLP, and Explainable AI for Cyber Investigations

This notebook is the streamlined lab for Session 5, ending with LIME and SHAP interpretation. It assumes **no coding background**. Every section matches an Activity in the printed Lab Manual — if anything here is unclear, the manual explains the same step in more detail.

**Rules of the bench:**
1. Run cells **top to bottom, in order**. The number in brackets `[4]` shows the order a cell actually ran — if your numbers are out of sequence, your results may not be reproducible. In forensic work, that matters.
2. Runtime type: **CPU is enough.** Do not switch to GPU (`Runtime ▸ Change runtime type`).
3. If the runtime disconnects or dies: `Runtime ▸ Restart session`, then use the **Rescue cell** in the Appendix to rebuild everything in one go.

---

## SECTION 0 · Set up the bench  *(≈ 1 minute, run this first)*
Colab machines already have `pandas`, `scikit-learn`, `nltk`, `spacy`, and `matplotlib` installed. We add the two explanation libraries, then fetch the small language resources NLTK and spaCy need.

`!` at the start of a line means: *this is a command to the machine itself, not Python.*

In [ ]:
!pip install -q lime shap
print("Libraries installed.")

In [ ]:
# Language resources for NLTK (tokeniser + stopword list)
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

# spaCy's small English model — download only if it is not already present
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])
    nlp = spacy.load("en_core_web_sm")

print("Bench ready.")

## SECTION 1 · First contact  *(for anyone who has never written code)*
Three tiny cells. If you have coded before, run them anyway — they take ten seconds and confirm your runtime is alive.

In [ ]:
# A code cell runs when you press Shift+Enter (or the ▶ button on its left).
print("Hello. This machine is not my laptop — it is a computer in a Google data centre.")

In [ ]:
# Variables are labelled boxes. Put a value in; use it later by name.
investigator = "your name here"       # <-- edit this, then re-run the cell
case_id = "ISEA-S5-001"
print("Examiner:", investigator, "| Case:", case_id)

In [ ]:
# A list holds several values; a loop visits each one.
sample_messages = [
    "Congratulations! You won a FREE prize. Call now!",
    "ok bro, reaching campus by 5, wait near the gate",
    "URGENT: your account will be suspended, verify immediately",
]
for m in sample_messages:
    print(len(m), "characters ->", m)

**Deliberate error (do not skip).** Run the next cell. It will fail, on purpose — read the last line of the red output before fixing anything.

In [ ]:
# This cell FAILS on purpose: 'evidence_count' was never defined.
print(evidence_count)

The last line says `NameError: name 'evidence_count' is not defined`. Python errors read **bottom-up**: the last line is the diagnosis, everything above it is the trail. Fix it by defining the name first — run the next cell.

In [ ]:
evidence_count = 3
print(evidence_count)   # same statement as before — now it works, because the name exists

## SECTION 2 · The evidence  *(load the corpus)*
**The case.** A fraud team hands you an export of **5,574 SMS messages**, each already labelled `ham` (normal traffic) or `spam` (unsolicited/scam bait). Your job across this lab: build a first-pass triage tool, then — more importantly — work out whether you can trust it.

**Provenance, because provenance always comes first:** this is the *SMS Spam Collection* (Almeida & Gómez Hidalgo), a public research corpus hosted by the UCI Machine Learning Repository. Real messages, collected and labelled by researchers — not synthetic.

In [ ]:
import requests, zipfile, io

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
try:
    r = requests.get(url, timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall(".")
    print("Downloaded from UCI.")
except Exception as e:
    # Fallback mirror (same corpus, tab-separated) in case UCI is unreachable
    print("UCI unreachable, using mirror:", e)
    r = requests.get("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv", timeout=60)
    with open("SMSSpamCollection", "w") as f:
        f.write(r.text)
    print("Downloaded from mirror.")

In [ ]:
import pandas as pd

df = pd.read_csv("SMSSpamCollection", sep="\t", header=None,
                 names=["label", "message"], quoting=3)
print("Rows and columns:", df.shape)
df.head(8)

In [ ]:
# How is the evidence distributed?
print(df["label"].value_counts())
print()
print("Spam share: %.1f%%" % (100 * (df["label"] == "spam").mean()))

**Checkpoint.** You should see **5,574 rows**, split **4,827 ham / 747 spam** — about **13.4% spam**. Class imbalance is normal in forensic triage: the thing you are hunting is always the minority.

In [ ]:
# Read actual evidence — never model what you have not read.
print("--- three spam messages ---")
for m in df[df.label == "spam"]["message"].head(3):
    print("•", m, "\n")
print("--- three ham messages ---")
for m in df[df.label == "ham"]["message"].head(3):
    print("•", m, "\n")

## SECTION 3 · Indicators before models
Before any machine learning: what do scam messages *contain* that normal messages do not? We count four indicator types with simple pattern matching (regular expressions), the way an examiner would grep a log.

In [ ]:
indicators = {
    "contains URL":         r"(?:https?://|www\.)\S+",
    "long number (5-11 digits)": r"\b\d{5,11}\b",
    "money symbol/amount":  r"[£$€]\s?\d[\d,\.]*|\b\d+\s?p\b",
    "bait word":            r"\b(?:free|win|winner|urgent|claim|prize)\b",
}
print(f"{'indicator':28s}{'in spam':>10s}{'in ham':>10s}")
for name, pattern in indicators.items():
    hit = df["message"].str.contains(pattern, regex=True, case=False)
    print(f"{name:28s}{100*hit[df.label=='spam'].mean():9.1f}%{100*hit[df.label=='ham'].mean():9.1f}%")

caps = df["message"].str.count(r"\b[A-Z]{2,}\b")
print(f"\nALL-CAPS words per message — spam: {caps[df.label=='spam'].mean():.2f}   ham: {caps[df.label=='ham'].mean():.2f}")

**Checkpoint.** Long digit strings appear in ~**74.6%** of spam and ~**0.1%** of ham. Money symbols: ~**50.6%** vs ~**0.2%**. You have real signal before any model exists — remember this when the model later takes the credit.

Now the Session 2 debt: **NLTK** tokenisation and **spaCy** entity extraction on one real scam message.

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

evidence_msg = ("You have WON a guaranteed £1000 cash or a £2000 prize. "
                "To claim yr prize call our customer service representative on 09061701461.")

tokens = word_tokenize(evidence_msg)
print("Tokens:", tokens[:14], "...")

sw = set(stopwords.words("english"))
content = [t for t in tokens if t.lower() not in sw and t.isalpha()]
print("Content words only:", content)

In [ ]:
# spaCy named-entity recognition on the same message
doc = nlp(evidence_msg)
for ent in doc.ents:
    print(f"{ent.text:22s} -> {ent.label_}")

**Look closely at the last line.** spaCy tags `09061701461` — a premium-rate callback number, the operational heart of the scam — as **DATE**. The model was trained on news text; SMS-speak confuses it. *An NLP tool's output is a lead to verify, never a fact to file.* That sentence is the whole point of this section.

## SECTION 4 · Train the triage model
We convert each message to numbers (**TF-IDF**: which words it uses, weighted by how distinctive they are) and fit **logistic regression** — a simple, fast, *inspectable* classifier. We hold out 25% of the data the model never sees during training; that held-out set is the only honest test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"],
    test_size=0.25, random_state=42, stratify=df["label"])
print("Training messages:", len(X_train), "| Held-out test messages:", len(X_test))

model = make_pipeline(
    TfidfVectorizer(lowercase=True, stop_words="english"),
    LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)
print("Model trained.")

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred)*100:.2f}%\n")
print(classification_report(y_test, pred, digits=3))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred, labels=["ham", "spam"],
    display_labels=["ham (normal)", "spam (scam)"], cmap="Blues", colorbar=False)
plt.title("Held-out test set")
plt.show()

**Read this like an examiner, not a leaderboard.**
- Accuracy ≈ **97.3%** — the headline.
- False alarms (ham flagged as spam): **0**. No innocent message was accused.
- Misses (spam passed as ham): **38 of 187** — spam recall ≈ **79.7%**. *One scam in five walks straight through.*

Same model, same numbers, two different stories. Accuracy is a headline; the confusion matrix is the finding.

## SECTION 5 · The verdict — and the vote
The model now scores messages it has never seen. Run the next cell, then **stop**.

In [ ]:
new_messages = [
    "URGENT! You have won a free prize of £2000. Call 09061701461 now to claim.",
    "ok bro, reaching campus by 5, wait near the gate",
]
proba = model.predict_proba(new_messages)
spam_col = list(model.classes_).index("spam")
for msg, p in zip(new_messages, proba):
    print(f"p(spam) = {p[spam_col]*100:5.1f}%   {msg}")

In [ ]:
# One real held-out message the model is ~97% sure about:
import numpy as np
p_test = model.predict_proba(X_test)[:, spam_col]
i = int(np.argsort(np.abs(p_test - 0.97))[0])
alert_msg = X_test.iloc[i]
print(f"p(spam) = {p_test[i]*100:.1f}%")
print("MESSAGE:", alert_msg)

### ✋ THE VOTE — write before you scroll
Your model — the one *you* trained four cells ago — says **~97% spam** on that message. There is a queue behind it, and no explanation attached.

In your worksheet (Section 8), write one of: **ACT / DON'T / NEED MORE** — and if NEED MORE, write *exactly what* you would need to see. Do it now. The rest of this notebook is the answer to whatever you just wrote.

## SECTION 6 · LIME — Exercise 1: explain one alert
LIME never looks inside the model. It perturbs this one message hundreds of times — dropping words — watches the score move, and fits a small readable model to that neighbourhood. What you get: the words that pushed *this* prediction, with direction and weight.

In [ ]:
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=list(model.classes_), random_state=42)
exp = explainer.explain_instance(alert_msg, model.predict_proba,
                                 num_features=8, num_samples=1000)

from IPython.display import HTML
HTML(exp.as_html())   # rich view: class probabilities + the message with words highlighted

In [ ]:
# The same result as plain numbers (positive = pushed towards spam)
for word, weight in exp.as_list():
    print(f"{word:14s} {weight:+.3f}")

**Exercise 1 — answer in the worksheet:**
1. Which words pushed the verdict towards *spam*? Would you defend each one to a sceptical colleague, out loud?
2. Notice `STOP` near the top. UK bulk senders are required to include opt-out wording — the model has learned *compliance boilerplate* as a scam marker. Is that evidence about the message, or about the sender's mailing software?
3. **Stability test** — run the next cell (different seed, fewer samples). Does your top-5 list survive?

In [ ]:
exp2 = LimeTextExplainer(class_names=list(model.classes_), random_state=7) \
        .explain_instance(alert_msg, model.predict_proba, num_features=8, num_samples=200)
for word, weight in exp2.as_list():
    print(f"{word:14s} {weight:+.3f}")

top1 = {w for w, _ in exp.as_list()[:5]}
top2 = {w for w, _ in exp2.as_list()[:5]}
print("\nTop-5 overlap:", len(top1 & top2), "of 5. Changed:", top1 ^ top2)

If your ranking moved (typically one of five features swaps), that is not a bug — it is the approximation cost of a sampled method. **If a ranking is not stable, you have a measurement, not a finding.** Report the seed and sample size, or do not report the ranking.

## SECTION 7 · SHAP — Exercise 2: local *and* global
SHAP splits the credit for a prediction across features using a fairness rule from game theory. Its values **sum exactly** to the gap between this prediction and the model's baseline — which is why SHAP plots reconcile and LIME's do not.

In [ ]:
import shap

vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]
X_train_v = vec.transform(X_train)
X_test_v  = vec.transform(X_test)
feature_names = list(vec.get_feature_names_out())

shap_explainer = shap.LinearExplainer(clf, X_train_v)
sv = shap_explainer(X_test_v)
sv.feature_names = feature_names
print("SHAP values computed for", sv.values.shape[0], "test messages ×", sv.values.shape[1], "features")

In [ ]:
# LOCAL — waterfall for the alert message (one prediction, decomposed)
one = shap.Explanation(
    values=np.asarray(sv.values[i].todense()).ravel() if hasattr(sv.values[i], "todense") else sv.values[i],
    base_values=float(sv.base_values[i]) if np.ndim(sv.base_values) else float(sv.base_values),
    data=np.asarray(X_test_v[i].todense()).ravel(),
    feature_names=feature_names)
shap.plots.waterfall(one, max_display=10)

In [ ]:
# GLOBAL — beeswarm across a 300-message sample of the test set
rng = np.random.RandomState(0)
samp = rng.choice(X_test_v.shape[0], 300, replace=False)
vals = sv.values[samp]
vals = np.asarray(vals.todense()) if hasattr(vals, "todense") else vals
sub = shap.Explanation(
    values=vals,
    base_values=np.full(300, float(sv.base_values if np.ndim(sv.base_values) == 0 else sv.base_values[0])),
    data=np.asarray(X_test_v[samp].todense()),
    feature_names=feature_names)
shap.plots.beeswarm(sub, max_display=12)

**Exercise 2 — answer in the worksheet:**
1. Compare the waterfall's top features with LIME's list from Exercise 1. Where do the two methods agree? *Only the features both agree on belong in a report.*
2. Look at the beeswarm's top rows. **`lt` and `gt` are fragments of `&lt;` and `&gt;`** — HTML escape codes left behind by how this corpus was prepared. Two of your model's most influential features are packaging artefacts, not language. You did not plant this; it was already there. Would you be comfortable defending a model that relies heavily on these artefacts? Why or why not?
3. Write the honest sentence: *"the model's decision on this message was driven principally by ___ and ___"* — model behaviour, not world causation.

## SECTION 8 · Worksheet and close-out
Copy this into your record (or answer directly below in a text cell of your own).

**The vote (Section 5):** ACT / DON'T / NEED MORE — and what, exactly, you needed.

**Exercise 1 (LIME):**
- Words I would defend, out loud: …
- Words I would not: …
- Stability: my top-5 overlap was ___ of 5; what moved: …

**Exercise 2 (SHAP):**
- Features LIME and SHAP agree on: …
- Features I would question or investigate further: …
- The honest sentence: *"the model's decision on this message was driven principally by ___ and ___."*

**Final reflection:**
- What did LIME reveal about this individual prediction?
- What did SHAP reveal about the model overall?
- Would you act on the original 97% alert now? Explain your answer.

**One sentence to keep:** an attribution tells you what *this model used* — not what causes scams, and not what would change in the world if a word changed.


---
## Appendix · Rescue cell
Runtime died? `Runtime ▸ Restart session`, then run **only this cell** — it rebuilds the bench and the model in one pass (≈ 2 minutes), after which you can jump back to whichever section you were in.

In [ ]:
!pip install -q lime shap
import requests, zipfile, io, nltk, spacy, numpy as np, pandas as pd
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"]); nlp = spacy.load("en_core_web_sm")
try:
    r = requests.get("https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall(".")
except Exception:
    r = requests.get("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv", timeout=60)
    open("SMSSpamCollection", "w").write(r.text)
df = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label", "message"], quoting=3)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
X_train, X_test, y_train, y_test = train_test_split(df["message"], df["label"], test_size=0.25, random_state=42, stratify=df["label"])
model = make_pipeline(TfidfVectorizer(lowercase=True, stop_words="english"), LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)
spam_col = list(model.classes_).index("spam")
p_test = model.predict_proba(X_test)[:, spam_col]
i = int(np.argsort(np.abs(p_test - 0.97))[0]); alert_msg = X_test.iloc[i]
print(f"Rescued. Model accuracy: {accuracy_score(y_test, model.predict(X_test))*100:.2f}% | alert message reloaded.")